# Drosophila Serial-Section Alignment

Split an annotated 3D Drosophila dataset into ordered sections, estimate serial transformations, and reconstruct the aligned point cloud.

This curated notebook targets the current Dynamo-free Spateo API. Edit the configuration cell before execution.


## Configure the Drosophila serial-section experiment


In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import spateo as st
import torch

warnings.filterwarnings("ignore")
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Spateo {st.__version__}; alignment device: {DEVICE}")

import anndata as ad
import matplotlib.pyplot as plt

INPUT_H5AD = Path("/DATA/User/gaomohan/Alignment/E16-18h_a_count_normal_stereoseq.h5ad")
ANNOTATION_KEY = "annotation"
RAW_3D_KEY = "spatial_3d"
SLICE_2D_KEY = "spatial_2d"
ALIGN_KEY = "align_spatial"
APPLY_SYNTHETIC_ROTATION = False


## Load, validate, normalize, and calculate PCA


In [ ]:
adata = st.read_h5ad(INPUT_H5AD)
required_columns = ["new_x", "new_y", "new_z"]
missing_columns = [column for column in required_columns if column not in adata.obs]
if missing_columns:
    raise KeyError(f"Missing coordinate columns: {missing_columns}")
if ANNOTATION_KEY not in adata.obs:
    raise KeyError(f"Missing obs[{ANNOTATION_KEY!r}].")
if not adata.obs_names.is_unique:
    raise ValueError("Observation identifiers must be unique before serial-section alignment.")

adata.obsm[RAW_3D_KEY] = adata.obs[required_columns].to_numpy(dtype=float, copy=True)
if not np.isfinite(adata.obsm[RAW_3D_KEY]).all():
    raise ValueError("The reconstructed 3D coordinates contain non-finite values.")

if "counts_X" not in adata.layers:
    source_layer = next(
        (candidate for candidate in ("counts", "raw_counts") if candidate in adata.layers),
        None,
    )
    if source_layer is None:
        warnings.warn("No count layer found; treating X as counts. Verify this assumption.")
    adata.layers["counts_X"] = (
        adata.layers[source_layer].copy() if source_layer is not None else adata.X.copy()
    )
st.pp.normalize_total(
    adata,
    layer="counts_X",
    out_layer="norm_X",
    target_sum=None,
    size_factor_key="Size_Factor",
    inplace=True,
)
st.pp.log1p_layer(
    adata,
    layer="norm_X",
    out_layer="log1p_X",
    set_X=True,
    inplace=True,
)
st.pp.pca(adata, layer="log1p_X", pca_key="X_pca", n_pca_components=50, inplace=True)


## Split the 3D object into ordered 2D sections


In [ ]:
z_values = np.sort(np.unique(adata.obsm[RAW_3D_KEY][:, 2]))
slices = []
for z_value in z_values:
    section = adata[adata.obsm[RAW_3D_KEY][:, 2] == z_value].copy()
    section.obsm[SLICE_2D_KEY] = section.obsm[RAW_3D_KEY][:, :2].copy()
    slices.append(section)

if APPLY_SYNTHETIC_ROTATION:
    for section in slices[1:]:
        st.align.rigid_transformation(
            section,
            spatial_key=SLICE_2D_KEY,
            key_added=SLICE_2D_KEY,
            inplace=True,
        )

labels = adata.obs[ANNOTATION_KEY].astype("category").cat.categories.tolist()
colors = plt.cm.rainbow(np.linspace(0, 1, len(labels)))
palette = dict(zip(labels, colors))


## Review the input sections


In [ ]:
st.pl.slices_2d(
    slices=slices,
    label_key=ANNOTATION_KEY,
    spatial_key=SLICE_2D_KEY,
    height=2,
    center_coordinate=True,
    show_legend=True,
    ncols=5,
    palette=palette,
)


## Estimate and apply serial transformations


In [ ]:
transformations = st.align.morpho_align_transformation(
    models=slices,
    rep_layer="X_pca",
    rep_field="obsm",
    dissimilarity="cos",
    spatial_key=SLICE_2D_KEY,
    key_added=ALIGN_KEY,
    device=DEVICE,
    verbose=False,
)

aligned_slices = st.align.morpho_align_apply_transformation(
    models=slices,
    spatial_key=SLICE_2D_KEY,
    key_added=ALIGN_KEY,
    transformation=transformations,
)


## Review the aligned sections


In [ ]:
st.pl.slices_2d(
    slices=aligned_slices,
    label_key=ANNOTATION_KEY,
    spatial_key=ALIGN_KEY,
    height=2,
    center_coordinate=False,
    show_legend=True,
    ncols=5,
    palette=palette,
)


## Reconstruct the aligned 3D point cloud


In [ ]:
aligned_adata = ad.concat(aligned_slices, label="slice_index")
aligned_adata.obsm["aligned_spatial_3d"] = np.column_stack(
    [aligned_adata.obsm[ALIGN_KEY], aligned_adata.obsm[RAW_3D_KEY][:, 2]]
)

aligned_pc, point_palette = st.tdr.construct_pc(
    adata=aligned_adata,
    spatial_key="aligned_spatial_3d",
    groupby=ANNOTATION_KEY,
    key_added="tissue",
    colormap=palette,
)
st.pl.three_d_plot(
    model=aligned_pc,
    key="tissue",
    model_style="points",
    model_size=8,
    colormap=point_palette,
    show_axes=True,
    jupyter="static",
    window_size=(1200, 1200),
    show_outline=True,
    outline_kwargs={"show_labels": False, "outline_width": 3},
)
